In [2]:
#!/usr/bin/env python3
import requests
import urllib.parse
import pandas as pd
from io import StringIO
import importlib
from equity_data_importers import config

importlib.reload(config)
from equity_data_importers.config import Config
from sklearn.preprocessing import RobustScaler

import time
import requests


def fetch_gdelt_metric(query, start_date, end_date, geo, mode, value_name, retry_delay=5) -> pd.DataFrame:
    """
    Generic fetch for GDELT timelines with rate limiting.
    """
    print(f"Fetching GDELT {mode} for: {query}")
    params = {
        "query": query,
        "mode": mode,
        "format": "csv",
        "startdatetime": start_date.strftime("%Y%m%d") + "000000",
        "enddatetime": end_date.strftime("%Y%m%d") + "000000"
    }
    if geo:
        params["geo"] = geo

    url = "https://api.gdeltproject.org/api/v2/doc/doc?" + urllib.parse.urlencode(params)

    max_retries = 3
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, timeout=30)
            resp.raise_for_status()
            break
        except requests.exceptions.HTTPError as e:
            if resp.status_code == 429:
                wait_time = retry_delay * (2 ** attempt)  # exponential backoff
                print(f"⏳ Rate limit exceeded. Czekam {wait_time}s przed ponowną próbą...")
                time.sleep(wait_time)
            else:
                raise

    text = resp.text
    lines = text.splitlines()
    if lines and lines[0].startswith("sep="):
        lines = lines[1:]
    cleaned = "\n".join(lines)

    df = pd.read_csv(StringIO(cleaned), header=0, parse_dates=["Date"], index_col="Date")
    if "Series" in df.columns:
        df = df.drop(columns=["Series"])
    df.rename(columns={"Value": value_name}, inplace=True)

    # Czekaj między zapytaniami
    time.sleep(retry_delay)
    return df


def main():
    query = Config.GDELT_QUERY
    start_date = Config.START_DATE
    end_date = Config.END_DATE
    geo = getattr(Config, "GEO", None)

    # 1) Volume (% of all articles)
    vol = fetch_gdelt_metric(query, start_date, end_date, geo,
                             mode="TimelineVol", value_name="gdelt_articles")

    # 2) Tone (average tone metric)
    tone = fetch_gdelt_metric(query, start_date, end_date, geo,
                              mode="TimelineTone", value_name="sentiment_score")

    # 3) Robust-scale the volume
    scaler = RobustScaler()
    vol["gdelt_robust"] = scaler.fit_transform(vol[["gdelt_articles"]])

    # 4) Join them all together
    df = vol.join(tone, how="left")

    # 5) Write to the same file name
    output_file = "data/gdelt_data.csv"
    df.to_csv(output_file, index=True)
    print(f"Dane zapisane do: {output_file}")


if __name__ == "__main__":
    main()


🌐 Fetching GDELT TimelineVol for: (Tesla OR TSLA)
🌐 Fetching GDELT TimelineTone for: (Tesla OR TSLA)
💾 Dane zapisane do: data/gdelt_data.csv


In [ ]:
# RobustScaler is a data‐preprocessing transformer in scikit-learn that scales features using statistics that are robust to outliers. Instead of subtracting the mean and dividing by the standard deviation (as with a standard “z-score”), it:

# Subtracts the median of each feature,

# Divides by the interquartile range (IQR)—that is, the difference between the 75th and 25th percentiles.